In [1]:
import pandas as pd
import re

MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(lambda x: re.sub(r'_\d+$', '', x))
chr_lookup = manifest_df.set_index("core_name")["Chr"]
pos_lookup_m = manifest_df.set_index("core_name")["MapInfo"]

def get_chr(pid):
    return str(chr_lookup.get(re.sub(r'_\d+$', '', pid), "?"))

def get_pos(pid):
    val = pos_lookup_m.get(re.sub(r'_\d+$', '', pid), None)
    return int(val) if pd.notna(val) else None

# AA CPD network SNPs
aa_cpd_snps = [
    "exm2277017-0_T_R_1989215336",  # PPP1R12B Chr1
    "exm1567342-0_B_F_1918398641",  # DNAJC28 Chr21
    "exm318398-0_B_F_1922126681",   # MAPKAPK3 Chr3
    "exm469871-0_B_F_1921101970",   # NUDT12 Chr5
    "exm479357-0_T_R_1921130751"    # TGFBI Chr5
]

# EA CPD network SNPs
ea_cpd_snps = [
    "exm24035-0_B_R_1921346506",    # ARHGEF10L Chr1
    "exm1614640-0_T_R_1919086982",  # ARFGAP3 Chr22
    "exm659391-0_B_R_1918526444",   # CALD1 Chr7
    "exm776310-0_B_R_1922396242"    # PAPPA Chr9
]

print("AA CPD SNPs:")
for pid in aa_cpd_snps:
    print(f"  Chr{get_chr(pid)}:{get_pos(pid):,}  {pid[:35]}")

print("\nEA CPD SNPs:")
for pid in ea_cpd_snps:
    print(f"  Chr{get_chr(pid)}:{get_pos(pid):,}  {pid[:35]}")

# overlap analysis
def find_overlap(aa_ids, ea_ids, window_bp):
    shared = []
    for aa_pid in aa_ids:
        aa_chr, aa_p = get_chr(aa_pid), get_pos(aa_pid)
        for ea_pid in ea_ids:
            ea_chr, ea_p = get_chr(ea_pid), get_pos(ea_pid)
            if aa_chr == ea_chr and aa_p and ea_p:
                dist = abs(aa_p - ea_p)
                if dist <= window_bp:
                    shared.append({
                        "AA_probe": aa_pid,
                        "EA_probe": ea_pid,
                        "Chr": aa_chr,
                        "AA_pos": aa_p,
                        "EA_pos": ea_p,
                        "distance_bp": dist
                    })
    return pd.DataFrame(shared)

print("\n=== CPD CROSS-ANCESTRY OVERLAP ===")
for window in [100_000, 500_000, 1_000_000, 5_000_000]:
    shared = find_overlap(aa_cpd_snps, ea_cpd_snps, window)
    print(f"Within {window//1000}kb: {len(shared)} shared signals")
    if len(shared) > 0:
        for _, row in shared.iterrows():
            print(f"  Chr{row['Chr']}: AA={row['AA_pos']:,} | EA={row['EA_pos']:,} | dist={row['distance_bp']:,}bp")

# chromosome comparison
aa_chroms = set(get_chr(p) for p in aa_cpd_snps)
ea_chroms = set(get_chr(p) for p in ea_cpd_snps)
print(f"\nAA chromosomes: {sorted(aa_chroms)}")
print(f"EA chromosomes: {sorted(ea_chroms)}")
print(f"Shared chromosomes: {sorted(aa_chroms & ea_chroms)}")

AA CPD SNPs:
  Chr1:202,399,880  exm2277017-0_T_R_1989215336
  Chr21:34,860,677  exm1567342-0_B_F_1918398641
  Chr3:50,614,990  exm318398-0_B_F_1922126681
  Chr5:102,895,930  exm469871-0_B_F_1921101970
  Chr5:135,392,482  exm479357-0_T_R_1921130751

EA CPD SNPs:
  Chr1:17,982,446  exm24035-0_B_R_1921346506
  Chr22:43,218,397  exm1614640-0_T_R_1919086982
  Chr7:134,617,884  exm659391-0_B_R_1918526444
  Chr9:119,028,233  exm776310-0_B_R_1922396242

=== CPD CROSS-ANCESTRY OVERLAP ===
Within 100kb: 0 shared signals
Within 500kb: 0 shared signals
Within 1000kb: 0 shared signals
Within 5000kb: 0 shared signals

AA chromosomes: ['1', '21', '3', '5']
EA chromosomes: ['1', '22', '7', '9']
Shared chromosomes: ['1']
